**Notebook Context and Continuity**

This notebook is a continuation of the core emotion classification model developed in the primary modeling notebook. While the previous notebook focuses on model development and training, this notebook builds on that foundation to further analyze results, generate predictions, and translate model outputs into interpretable insights.

The model architecture, training process, and key preprocessing steps are assumed to be established in the preceding notebook and are not redefined in full here.


**Check Saved Model Directory**

In [ ]:
!ls emotion_model


ls: cannot access 'emotion_model': No such file or directory


**Environment Setup**

In [ ]:
!pip install flask pyngrok transformers torch --quiet

**Load Fine-Tuned Model + Set Emotion–Color Mapping**

In [ ]:
!unzip -o "/content/emotion_model (1).zip" -d /content/
!ls -F emotion_model/

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load your saved model
MODEL_PATH = "emotion_model/"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Confirm label order
LABEL_MAP = model.config.id2label
print("MODEL LABEL MAP:", LABEL_MAP)

# YOUR 8 emotions mapped correctly
ID_TO_EMOTION = {
    "LABEL_0": "anger",
    "LABEL_1": "disgust",
    "LABEL_2": "fear",
    "LABEL_3": "surprise",
    "LABEL_4": "joy",
    "LABEL_5": "love",
    "LABEL_6": "neutral",
    "LABEL_7": "sadness"
}

# Exact 8-colors (no brightness changes)
EMOTION_COLORS = {
    "anger": "#FF0000",
    "disgust": "#556B2F",
    "fear": "#8A2BE2",
    "surprise": "#FFA500",
    "joy": "#00FF00",
    "love": "#FF69B4",
    "neutral": "#808080",
    "sadness": "#0000FF"
}

print("\nAll mappings loaded successfully.")

Archive:  /content/emotion_model (1).zip
   creating: /content/emotion_model/
  inflating: /content/emotion_model/merges.txt  
  inflating: /content/emotion_model/special_tokens_map.json  
  inflating: /content/emotion_model/vocab.json  
  inflating: /content/emotion_model/config.json  
  inflating: /content/emotion_model/tokenizer.json  
  inflating: /content/emotion_model/training_args.bin  
  inflating: /content/emotion_model/model.safetensors  
  inflating: /content/emotion_model/tokenizer_config.json  
config.json  model.safetensors	      tokenizer_config.json  training_args.bin
merges.txt   special_tokens_map.json  tokenizer.json	     vocab.json
MODEL LABEL MAP: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7'}

All mappings loaded successfully.


**Emotion Prediction Function**

This function takes a text message as input, tokenizes it, and uses the fine-tuned model to predict one of the 8 supported emotions.
It also:

Converts the raw model label (e.g., "LABEL_3") into a human-readable emotion

Identifies the confidence score of the prediction

Assigns the matching HEX color for UI visualization



In [ ]:
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=1)
    conf, pred_id = torch.max(probs, dim=1)

    raw_label = LABEL_MAP[pred_id.item()]
    emotion = ID_TO_EMOTION[raw_label]
    color = EMOTION_COLORS[emotion]

    print("\nTEXT:", text)
    print("RAW LABEL:", raw_label)
    print("MAPPED EMOTION:", emotion)
    print("COLOR:", color)

    return emotion, conf.item(), color


**Mobile Chat UI — HTML Layout Template**

In [ ]:
HTML = """
<!DOCTYPE html>
<html>
<head>
<title>Emotion Chat</title>
<style>
body {
  background:#111; font-family:Arial; display:flex; justify-content:center; padding:20px;
}
.phone {
  width:380px; height:700px; background:black; border-radius:35px; padding:20px; color:white;
}
.header { text-align:center; font-size:24px; }
.chat-box {
  height:520px; overflow-y:auto; padding:10px;
}
.bubble {
  padding:12px; border-radius:18px; margin:10px;
  max-width:70%; color:white; font-size:15px;
}
.you { margin-left:auto; text-align:right; }
.them { margin-right:auto; }
</style>
</head>

<body>
<div class="phone">
  <div class="header">Alex</div>

  <div id="chat" class="chat-box"></div>

  <div style="display:flex; gap:8px;">
    <input id="you_msg" placeholder="You..." style="flex:1; padding:10px; border-radius:15px;">
    <button onclick="sendYou()">Send</button>
  </div>

  <div style="display:flex; gap:8px; margin-top:8px;">
    <input id="them_msg" placeholder="Them..." style="flex:1; padding:10px; border-radius:15px;">
    <button onclick="sendThem()">Send</button>
  </div>
</div>

<script>
async function send(type) {
  const box = type === "you" ? "you_msg" : "them_msg";
  const text = document.getElementById(box).value;
  if (!text) return;

  const res = await fetch("/predict", {
    method:"POST",
    headers:{"Content-Type":"application/json"},
    body:JSON.stringify({ text })
  });

  const data = await res.json();

  const div = document.createElement("div");
  div.className = "bubble " + type;
  div.innerText = text;
  div.style.backgroundColor = data.color;

  document.getElementById("chat").appendChild(div);
  document.getElementById(box).value = "";
}

function sendYou(){ send("you"); }
function sendThem(){ send("them"); }
</script>

</body>
</html>
"""


**Flask Web Server + Emotion Prediction API**

This section initializes a Flask web server to run the phone chat UI and connect it to the trained model.

In [ ]:
from flask import Flask, request, jsonify, render_template_string

app = Flask(__name__)

@app.route("/")
def home():
    return render_template_string(HTML)

@app.route("/predict", methods=["POST"])
def predict():
    text = request.json["text"]
    emotion, conf, color = predict_emotion(text)
    return jsonify({
        "emotion": emotion,
        "confidence": conf,
        "color": color
    })


**Secure Public Hosting with Ngrok + Flask Launch**

We use ngrok to create a secure public URL that exposes our local Flask server to the internet.
This lets the Emotion Chat UI run on a real phone for testing.

In [ ]:
ngrok.set_auth_token("36fSSd7dhCGeOpENh4U4RiyLJF0_VoifVSxkG73vmL6NCfrM")


# Start tunnel
public_url = ngrok.connect(5000)
print("🔗 PUBLIC NGROK URL:", public_url)

# Start Flask
app.run(port=5000)

🔗 PUBLIC NGROK URL: NgrokTunnel: "https://intromissible-ginger-nondurably.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [10/Dec/2025 22:17:36] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Dec/2025 22:59:25] "POST /predict HTTP/1.1" 200 -



TEXT: How is your day doing?
RAW LABEL: LABEL_6
MAPPED EMOTION: neutral
COLOR: #808080


INFO:werkzeug:127.0.0.1 - - [10/Dec/2025 22:59:39] "POST /predict HTTP/1.1" 200 -



TEXT: My day has been really great!
RAW LABEL: LABEL_4
MAPPED EMOTION: joy
COLOR: #00FF00


INFO:werkzeug:127.0.0.1 - - [10/Dec/2025 22:59:45] "POST /predict HTTP/1.1" 200 -



TEXT: And your?
RAW LABEL: LABEL_3
MAPPED EMOTION: surprise
COLOR: #FFA500


INFO:werkzeug:127.0.0.1 - - [10/Dec/2025 23:00:08] "POST /predict HTTP/1.1" 200 -



TEXT: I had alot of trafic this moring and i was late for work
RAW LABEL: LABEL_7
MAPPED EMOTION: sadness
COLOR: #0000FF
